## makemore: part 5 (building a WaveNet)

[DeepMind blog post from 2016](https://www.deepmind.com/blog/wavenet-a-generative-model-for-raw-audio)

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
import seaborn as sns
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [3]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [4]:
# shuffle up the words
import random
random.seed(42)
random.shuffle(words)

In [133]:
# build the dataset
block_size = 8 # context length: how many characters do we take to predict the next one?

def build_dataset(words): 
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix] # crop and append

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182441, 8]) torch.Size([182441])
torch.Size([22902, 8]) torch.Size([22902])
torch.Size([22803, 8]) torch.Size([22803])


In [134]:
for x,y in zip(Xtr[:20], Ytr[:20]):
    print(''.join(itos[ix.item()] for ix in x), '-->', itos[y.item()])

........ --> e
.......e --> l
......el --> i
.....eli --> a
....elia --> n
...elian --> y
..eliany --> s
.elianys --> .
........ --> t
.......t --> r
......tr --> o
.....tro --> y
....troy --> .
........ --> m
.......m --> a
......ma --> r
.....mar --> k
....mark --> u
...marku --> s
..markus --> .


In [141]:
Xtr.shape

torch.Size([182441, 8])

In [126]:
class Linear: 
    def __init__(self, fan_in, fan_out, bias=True): 
        self.weight = torch.randn((fan_in, fan_out)) /fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None
                                
    def __call__(self, x): 
        self.out = x @ self.weight 
        if self.bias is not None: 
            self.out += self.bias
        return self.out
    
    def parameters(self):
        return [self.weight] + ([self.bias] if self.bias is not None else [])

     
    def __repr__(self): 
        if self.bias is not None: 
            return f"Layer({self.weight.shape[0]}, {self.weight.shape[1]}), Bias({self.bias.shape[0]})"
        else: 
            return f"Layer({self.weight.shape[0]}, {self.weight.shape[1]})"
        

class BatchNorm1d: 
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        
        self.gamma = torch.ones(dim) #bngain
        self.beta = torch.zeros(dim)  #bnbias
        
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)
        
    def __call__(self, x): 
        if self.training: 
            if x.ndim ==2: 
                dim = 0
            elif x.ndim == 3: 
                dim = (1, 0)
            else: 
                raise("Unknow dimensions")
            mean = x.mean(dim, keepdims=True) # batch mean
            var  = x.var(dim, keepdims=True) # batch variance
            self.out = self.gamma * ( (x-mean) / (torch.sqrt(var+self.eps)) )  + self.beta
            
            with torch.no_grad(): 
                self.running_mean = ((1-self.momentum) * self.running_mean) + (self.momentum * mean)
                self.running_var = ((1-self.momentum) * self.running_var) + (self.momentum * var)
                
            return self.out
        
        else: 
            self.out = self.gamma * ( (x-self.running_mean) / (torch.sqrt(self.running_var+self.eps)) )  + self.beta
            return self.out
        
    def parameters(self): 
        return [self.gamma, self.beta]
    
    def __repr__(self): 
        return f"BatchNorm1d({self.gamma.shape[0]})"
    
    
class Tanh:         
    def __call__(self, x): 
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self): 
        return []
    
    def __repr__(self): 
        return "Tanh()"
    
    
class Embedding: 
    def __init__(self, num_embeddings, embeddings_dim): 
        self.weight = torch.randn(num_embeddings, embeddings_dim)
        
    def __call__(self, ix): 
        self.out = self.weight[ix]
        return self.out
    
    def __repr__(self): 
        return f"Embedding({self.weight.shape[0]}, {self.weight.shape[1]})"
    
    def parameters(self): 
        return [self.weight]
    

class FlattenConsecutive: # Different from pytorch API
    def __init__(self, n): # numnber of consecutive characters
        self.n = n
        
    def __call__(self, x): 
        B, R, C = x.shape
        x = x.view(B, R//self.n, C*self.n)
        
        if x.shape[1] == 1: # Useless dimension, (4, 1, 80) --> (4, 80) if n is high enough
            x = x.squeeze(1) # or x.view(B, C*n)
    
        self.out = x
        return self.out
    def parameters(self): 
        return []
    def __repr__(self): 
        return "Flatten()"
    
    
class Sequential: 
    def __init__(self, layers): 
        self.layers = layers
        
    def __call__(self, x): 
        for layer in self.layers: 
            x = layer(x)
        self.out = x
        return self.out
    
    def parameters(self): 
        return [p for layer in self.layers for p in layer.parameters()]

In [127]:
torch.manual_seed(42); # seed rng for reproducibility

In [142]:
n_embd = 14
n_hidden = 100
block_size = 8

model = Sequential([
    Embedding(vocab_size, n_embd), 
    FlattenConsecutive(2),Linear(n_embd * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2),Linear(2*n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2),Linear(2*n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size)
])

with torch.no_grad():
    model.layers[-1].weight *= 0.1 
    
parameters = model.parameters()

print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True

46505


In [144]:
# same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):
  
    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
    
    logits = model(Xb)
    loss = F.cross_entropy(logits, Yb) # loss function

    for p in parameters:
        p.grad = None
        
    loss.backward()

    # update
    lr = 0.1 if i < 150000 else 0.01 # step learning rate decay
    for p in parameters:
        p.data += -lr * p.grad

    # track stats
    if i % 10000 == 0: # print every once in a while
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

#     if i > 30000: 
#         break

      0/ 200000: 2.2629
  10000/ 200000: 2.2472
  20000/ 200000: 2.1356
  30000/ 200000: 2.1010
  40000/ 200000: 2.0787
  50000/ 200000: 1.8298
  60000/ 200000: 1.9527
  70000/ 200000: 2.2489
  80000/ 200000: 2.7511
  90000/ 200000: 2.2845
 100000/ 200000: 1.7058
 110000/ 200000: 2.1062
 120000/ 200000: 1.6884
 130000/ 200000: 2.2109
 140000/ 200000: 2.2257
 150000/ 200000: 2.1150
 160000/ 200000: 1.9984
 170000/ 200000: 1.6306
 180000/ 200000: 1.8067
 190000/ 200000: 1.4748


In [ ]:
for layer in model.layers: 
    print(layer.__class__.__name__, ":", tuple(layer.out.shape))

In [43]:
len(lossi)

30002

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(torch.tensor(lossi).view(-1, 1000).mean(1));

In [145]:
# put layers into eval mode (needed for batchnorm especially)
for layer in model.layers:
    layer.training = False

In [146]:
@torch.no_grad() 
def split_loss(split):
    
    x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
    }[split]
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')


train 1.823869228363037
val 1.9938013553619385


In [147]:
for _ in range(5):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
        # forward pass the neural net
        
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        # sample from the distribution
        ix = torch.multinomial(probs, num_samples=1).item()
        # shift the context window and track the samples
        context = context[1:] + [ix]
        out.append(ix)
        # if we sample the special '.' token, break
        if ix == 0:
            break

    print(''.join(itos[i] for i in out)) # decode and print the generated word



mckayla.
raky.
jahziel.
mazari.
sabelka.
